# Shell stationary heat problem
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meyer-nils/torch-fem/blob/main/examples/basic/shell/thermal_static.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/meyer-nils/torch-fem/main?labpath=examples/basic/shell/thermal_static.ipynb)

Compute the stationary temperature distribution in a curved shell.

In [1]:
import torch

from torchfem.data import get_data
from torchfem.io import import_shell
from torchfem.materials import IsotropicConductivity2D

# Set default data type to double precision
torch.set_default_dtype(torch.float64)

# Wall thickness t and vessel radius R
t = 6.0
R = 100.0

## Static heat equation on a curved surface

In [2]:
# Material model
material = IsotropicConductivity2D(kappa=6.0e-3)

# Import one octant of a pressure vessel, closed by three symmetry planes
vessel = import_shell(get_data("copv.vtu"), material, thickness=t)
nodes = vessel.nodes

# Hot boss at the dome apex
boss = nodes[:, 1:].norm(dim=1) < 0.2 * R
vessel.constraints[boss] = True
vessel.temperatures[boss, 0] = 150.0

# Cooled cradle at mid-length
cradle = nodes[:, 0] >= -0.001
vessel.constraints[cradle] = True
vessel.temperatures[cradle, 0] = 20.0

# The symmetry planes carry no flux, which needs no boundary condition

In [3]:
# Solve
temp, rfl, hf, temp_grad, _ = vessel.solve()

In [4]:
vessel.plot(node_property={"Temperature": temp}, colormap="magma")

EmbeddableWidget(value='<iframe srcdoc="<!doctype html>\n<html lang=&quot;en&quot;>\n  <head>\n    <meta chars…